# LangSmith Multi-Agent Trace Evaluation

This notebook is self-contained: it stores five synthetic traces, logs them to LangSmith, retrieves all root traces created within a configurable time window, evaluates each trace with six LLM-based metrics, and exports the results to JSON and Excel.

Run the cells from top to bottom. Re-running the logging cell publishes another set of five traces.


## 1. Install Dependencies

Installs the packages required for LangSmith trace logging and retrieval, Azure OpenAI evaluation, and Excel export.


In [ ]:
!pip install -q -U \
    langsmith \
    openai \
    pandas==2.2.2 \
    openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.6 MB/s eta 0:00:00


## 2. Configure Clients

Imports the required libraries and initializes the LangSmith and Azure OpenAI clients used throughout the notebook.


In [ ]:
import os
import json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd

from langsmith import Client
from langsmith.run_trees import RunTree
from openai import AzureOpenAI


# ============================================================
# LangSmith Configuration
# ============================================================

LANGSMITH_API_KEY = "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"
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
LANGSMITH_PROJECT = "multi-agent-demo"

os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_ENDPOINT"] = LANGSMITH_ENDPOINT
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT

langsmith_client = Client()


# ============================================================
# Azure OpenAI Configuration
# ============================================================

AZURE_OPENAI_ENDPOINT = "https://tcoeaiteamgpt41nano.openai.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2025-01-01-preview"
AZURE_OPENAI_API_KEY = "5xD6IeVNWL7eYqgbuxRKVATJK38hdacwDJnVT8rbKH9W1Kx6NhZZJQQJ99CDACYeBjFXJ3w3AAABACOGD59K"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-4.1-nano"

azure_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("✓ LangSmith client initialized")
print("✓ Azure OpenAI client initialized")


✓ LangSmith client initialized
✓ Azure OpenAI client initialized


## 3. Embedded Trace Data and Logging

Loads the baseline trace and four evaluation variants directly from the notebook. Run this cell to publish all five traces to the configured LangSmith project. Set `LOG_TRACES = False` to skip publishing and evaluate traces already stored in LangSmith instead.


In [ ]:
# ============================================================
# Embedded Trace Data
# ============================================================

EMBEDDED_TRACE_DATA = json.loads(r'''{
  "project": "multi-agent-demo",
  "start_time": "2026-07-10T09:00:00Z",
  "trace": {
    "name": "Supervisor Agent",
    "run_type": "chain",
    "duration": 0.2,
    "tags": [
      "supervisor",
      "orchestrator",
      "prod"
    ],
    "metadata": {
      "agent_role": "supervisor",
      "run_kind": "end_to_end_demo"
    },
    "inputs": {
      "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
    },
    "outputs": {
      "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
      "iterations": 2,
      "agents_used": [
        "research",
        "analysis",
        "report_writer"
      ],
      "approved": true
    },
    "children": [
      {
        "name": "Parameter Extraction Agent",
        "run_type": "llm",
        "duration": 0.7,
        "tags": [
          "supervisor",
          "parameter-extraction"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 182,
          "output_tokens": 112,
          "total_tokens": 294,
          "step": "parameter_extraction"
        },
        "inputs": {
          "messages": [
            {
              "role": "system",
              "content": "Extract structured planning parameters from the user's request."
            },
            {
              "role": "user",
              "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "parameters": {
            "company_profile": {
              "size": "mid-size",
              "origin": "US",
              "industry": "automotive"
            },
            "market": {
              "technology": "EV",
              "region": "Europe"
            },
            "target_year": 2027,
            "objective": "Recommend market-entry strategy",
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          }
        }
      },
      {
        "name": "Supervisor: Plan & Route",
        "run_type": "llm",
        "duration": 1.2,
        "tags": [
          "supervisor",
          "planning"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 420,
          "output_tokens": 96,
          "total_tokens": 516,
          "step": "plan"
        },
        "inputs": {
          "parameters": {
            "company_profile": {
              "size": "mid-size",
              "origin": "US",
              "industry": "automotive"
            },
            "market": {
              "technology": "EV",
              "region": "Europe"
            },
            "target_year": 2027,
            "objective": "Recommend market-entry strategy",
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "instruction": "Create an execution plan using the extracted planning parameters."
        },
        "outputs": {
          "role": "assistant",
          "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
          "route_to": "research_agent",
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          }
        }
      },
      {
        "name": "Research Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "research",
          "iteration-1"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor",
          "iteration": 1
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "task": "Gather evidence required by the execution plan."
        },
        "outputs": {
          "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "confidence": 0.82,
          "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
        },
        "children": [
          {
            "name": "Web Search Sub-Agent",
            "run_type": "tool",
            "duration": 2.4,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "tool": "web_search",
              "query": "Europe EV sales 2025 2026 by country market share"
            },
            "outputs": {
              "results": [
                "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                "Germany largest volume market; Norway/NL highest penetration."
              ]
            }
          },
          {
            "name": "Document Retrieval Sub-Agent",
            "run_type": "retriever",
            "duration": 1.8,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "query": "EU charging infrastructure density 2026 report"
            },
            "outputs": {
              "documents": [
                {
                  "source": "ACEA-2026.pdf",
                  "snippet": "~630k public charge points in EU."
                },
                {
                  "source": "IEA-EV-Outlook.pdf",
                  "snippet": "NL & DE lead fast-charger rollout."
                }
              ]
            }
          },
          {
            "name": "Source Validator Sub-Agent",
            "run_type": "chain",
            "duration": 1.1,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "claims": [
                "EU BEV share ~16%",
                "630k public charge points"
              ]
            },
            "outputs": {
              "validated": true,
              "notes": "Cross-checked ACEA + IEA; figures consistent."
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Research",
        "run_type": "llm",
        "duration": 0.9,
        "tags": [
          "supervisor",
          "review"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 512,
          "output_tokens": 60,
          "total_tokens": 572,
          "step": "review",
          "decision": "proceed"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Is the research sufficient to analyze?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Research solid. Route to Analysis Agent.",
          "route_to": "analysis_agent"
        }
      },
      {
        "name": "Analysis Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "analysis"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "task": "Analyze market size and competitors using research findings."
        },
        "outputs": {
          "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
          "confidence": 0.7
        },
        "children": [
          {
            "name": "Market Sizing Sub-Agent",
            "run_type": "tool",
            "duration": 2.0,
            "tags": [
              "sub-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "analysis_agent"
            },
            "inputs": {
              "tool": "market_model",
              "params": {
                "region": "EU",
                "segment": "C-segment BEV",
                "year": 2027
              }
            },
            "outputs": {
              "tam_units_2027": 2100000,
              "serviceable_units": 340000,
              "avg_price_eur": 38500
            }
          },
          {
            "name": "Competitor Benchmarking Sub-Agent",
            "run_type": "chain",
            "duration": 1.6,
            "tags": [
              "sub-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "analysis_agent"
            },
            "inputs": {
              "competitors": [
                "VW ID.3/ID.4",
                "Tesla Model 3/Y",
                "Renault",
                "MG"
              ]
            },
            "outputs": {
              "positioning": {
                "VW ID.3": "€40k, incumbent trust",
                "Tesla Model 3": "€42k, tech/brand",
                "MG": "€30k, price leader"
              },
              "gap": "mid-price with strong charging partnership underserved"
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Analysis",
        "run_type": "llm",
        "duration": 1.0,
        "tags": [
          "supervisor",
          "review",
          "loop-trigger"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 640,
          "output_tokens": 78,
          "total_tokens": 718,
          "step": "review",
          "decision": "loop_back"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Is the analysis complete enough to write?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
          "route_to": "research_agent"
        }
      },
      {
        "name": "Research Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "research",
          "iteration-2",
          "loop"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor",
          "iteration": 2
        },
        "inputs": {
          "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
        },
        "outputs": {
          "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
          "confidence": 0.86
        },
        "children": [
          {
            "name": "Web Search Sub-Agent",
            "run_type": "tool",
            "duration": 2.1,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "tool": "web_search",
              "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
            },
            "outputs": {
              "results": [
                "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                "Local battery content increasingly tied to purchase incentives.",
                "Hungary/Poland offer EV manufacturing incentives."
              ]
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Addendum",
        "run_type": "llm",
        "duration": 0.8,
        "tags": [
          "supervisor",
          "review"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 700,
          "output_tokens": 52,
          "total_tokens": 752,
          "step": "review",
          "decision": "proceed"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Do we now have enough to write?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Complete. Route to Report Writer Agent.",
          "route_to": "report_writer_agent"
        }
      },
      {
        "name": "Report Writer Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "writer"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
          "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "task": "Generate the final market-entry recommendation."
        },
        "outputs": {
          "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "word_count": 78
        },
        "children": [
          {
            "name": "Drafting Sub-Agent",
            "run_type": "llm",
            "duration": 2.7,
            "tags": [
              "sub-agent",
              "writer"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 1180,
              "output_tokens": 240,
              "total_tokens": 1420,
              "agent_role": "sub"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Write a concise strategy recommendation."
                },
                {
                  "role": "user",
                  "content": "Synthesize findings into a 2027 EU entry plan."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
            }
          },
          {
            "name": "Formatting & Citation Sub-Agent",
            "run_type": "chain",
            "duration": 1.0,
            "tags": [
              "sub-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "report_writer_agent"
            },
            "inputs": {
              "draft": "Recommended 2027 EU market-entry strategy...",
              "sources": [
                "ACEA-2026.pdf",
                "IEA-EV-Outlook.pdf"
              ]
            },
            "outputs": {
              "formatted": true,
              "citations_added": 2,
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
            }
          }
        ]
      },
      {
        "name": "Supervisor: Final Review",
        "run_type": "llm",
        "duration": 0.9,
        "tags": [
          "supervisor",
          "review",
          "final"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 900,
          "output_tokens": 40,
          "total_tokens": 940,
          "step": "final_review",
          "decision": "approve"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "messages": [
            {
              "role": "user",
              "content": "Approve the final report?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Approved. Report answers the query with data + regulatory context.",
          "approved": true
        }
      }
    ]
  },
  "traces": [
    {
      "id": "baseline",
      "description": "Original, fully aligned multi-agent trace.",
      "trace": {
        "name": "Supervisor Agent",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo"
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_A_poor_parameter_and_plan",
      "description": "Poor parameter extraction and plan accuracy.",
      "trace": {
        "name": "Supervisor Agent — variant_A_poor_parameter_and_plan",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_A_poor_parameter_and_plan"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_A_poor_parameter_and_plan",
          "variant_description": "Parameter extraction omits and misstates planning information; the planner follows it poorly."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "origin": "Europe"
                },
                "market": {
                  "technology": "hybrid",
                  "region": "North America"
                },
                "target_year": 2025,
                "objective": "Prepare a general product overview"
              },
              "content": "Extracted only partial parameters; several fields are uncertain."
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "origin": "Europe"
                },
                "market": {
                  "technology": "hybrid",
                  "region": "North America"
                },
                "target_year": 2025,
                "objective": "Prepare a general product overview"
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Prepare a generic North American hybrid overview for 2025 2) Draft a short product note. Route directly to Report Writer Agent.",
              "route_to": "report_writer_agent",
              "planning_context": {
                "planned_region": "North America",
                "planned_market": "hybrid",
                "planned_year": 2025,
                "deliverables": [
                  "General product overview"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_B_poor_trajectory_and_tools",
      "description": "Poor trajectory match and tool-call accuracy.",
      "trace": {
        "name": "Supervisor Agent — variant_B_poor_trajectory_and_tools",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_B_poor_trajectory_and_tools"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_B_poor_trajectory_and_tools",
          "variant_description": "Execution is routed out of order, skips validation/review stages, and uses unnecessary tools."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: skip research validation and draft the report first; perform analysis only if needed.",
              "route_to": "report_writer_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Currency Conversion Sub-Agent",
                "run_type": "tool",
                "inputs": {
                  "tool": "currency_converter",
                  "amount": 1,
                  "from": "EUR",
                  "to": "USD"
                },
                "outputs": {
                  "converted_amount": 1.08,
                  "note": "Not required for the requested strategy."
                },
                "duration": 0.2,
                "tags": [
                  "unnecessary-tool",
                  "variant-B"
                ],
                "metadata": {
                  "purpose": "Deliberately unnecessary call for evaluation."
                }
              }
            ]
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              }
            ]
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_C_poor_summary_accuracy",
      "description": "Poor summary accuracy from unsupported report claims.",
      "trace": {
        "name": "Supervisor Agent — variant_C_poor_summary_accuracy",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_C_poor_summary_accuracy"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_C_poor_summary_accuracy",
          "variant_description": "The report writer introduces unsupported and contradictory claims while upstream context is unchanged."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary.",
              "word_count": 78,
              "summary": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_D_poor_goal_accomplishment",
      "description": "Poor goal accomplishment despite otherwise correct execution.",
      "trace": {
        "name": "Supervisor Agent — variant_D_poor_goal_accomplishment",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_D_poor_goal_accomplishment"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_D_poor_goal_accomplishment",
          "variant_description": "The workflow executes normally, but the final response is generic and fails to address the requested strategy."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
              "word_count": 78,
              "summary": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved: a short generic market note was prepared.",
              "approved": true
            }
          }
        ]
      }
    }
  ]
}''')

print(f"Loaded {len(EMBEDDED_TRACE_DATA['traces'])} embedded trace(s).")


# ============================================================
# Log Embedded Traces to LangSmith
# ============================================================

class VirtualClock:
    """Creates increasing timestamps without waiting in real time."""

    def __init__(self, start_time):
        self.current_time = start_time

    def now(self):
        timestamp = self.current_time
        self.current_time += timedelta(milliseconds=50)
        return timestamp

    def advance(self, seconds):
        self.current_time += timedelta(seconds=float(seconds))
        return self.current_time


def emit_trace_node(node, parent, clock, project, client):
    extra = {"metadata": node["metadata"]} if node.get("metadata") else None
    run_arguments = {
        "name": node["name"],
        "run_type": node["run_type"],
        "inputs": node.get("inputs", {}),
        "start_time": clock.now(),
        "tags": node.get("tags", []),
        "extra": extra,
    }

    run = (
        RunTree(project_name=project, client=client, **run_arguments)
        if parent is None
        else parent.create_child(**run_arguments)
    )
    run.post()

    for child in node.get("children", []):
        emit_trace_node(child, run, clock, project, client)

    run.end(
        outputs=node.get("outputs", {}),
        end_time=clock.advance(node.get("duration", 0.5)),
    )
    run.patch()
    return run


def log_embedded_traces(trace_data=EMBEDDED_TRACE_DATA, project_name=LANGSMITH_PROJECT):
    """Publish every embedded trace and return its identifier and URL."""
    records = trace_data.get("traces", [])
    if not records:
        raise ValueError("Embedded trace data does not contain any traces.")

    logged_traces = []
    for record in records:
        root = emit_trace_node(
            record["trace"],
            parent=None,
            clock=VirtualClock(datetime.now(timezone.utc)),
            project=project_name,
            client=langsmith_client,
        )
        try:
            url = root.get_url()
        except Exception:
            url = None
        logged_traces.append({"variant_id": record["id"], "trace_id": str(root.id), "url": url})

    langsmith_client.flush()
    return logged_traces


LOG_TRACES = True

if LOG_TRACES:
    logged_traces = log_embedded_traces()
    print(f"Logged {len(logged_traces)} trace(s) to '{LANGSMITH_PROJECT}'.")
    for item in logged_traces:
        print(f"- {item['variant_id']}: {item['trace_id']}")
else:
    print("Logging skipped. Set LOG_TRACES = True to publish the embedded traces.")


## 4. Native Metric Prompts (ported verbatim from prompts.py)

Loads the exact registry prompts for the six metrics: the parameter/plan/trajectory `audit` + `overall_reason` prompts, the summary `claim_segmentation` + `claim_verification` prompts, the tool-call system prompt, and the goal-accomplishment builder. These are copied unchanged from `prompts.py`.

In [ ]:
# ============================================================
# Native Prompt Registry — ported VERBATIM from prompts.py
# ============================================================
# These are the exact registry prompts. Their original working is multi-stage
# and fragment-based (see next cell), not a single 1-10 judge call.

import json

NATIVE_PROMPTS = {
    "parameter_extraction": {
        "audit": '\nRole:\nYou are a STRICT Parameter Extraction Evaluation Engine.\nYou behave deterministically and mechanically.\n \nGoal:\nEvaluate extracted parameters using TWO SEQUENTIAL CHECKS.\n \nInputs:\nINPUT_TEXT:\n{INPUT_TEXT}\n \nEXTRACTED_PARAMETERS:\n{EXTRACTED_PARAMETERS}\n \nEXPECTED_PARAMETER_SCHEMA:\n{EXPECTED_PARAMETER_SCHEMA}\n \n------------------------------------------------------------\nEVALUATION FRAMEWORK (STRICT GATING LOGIC)\n------------------------------------------------------------\n \nYou MUST evaluate parameters using EXACTLY TWO checks:\n \n1. value_check\n2. format_check\n \nAdditionally you MUST output:\n- parameter_name\n- expected_value  (value grounded from INPUT_TEXT)\n- actual_value    (value from EXTRACTED_PARAMETERS)\n \n------------------------------------------------------------\nCHECK DEFINITIONS\n------------------------------------------------------------\n \nCHECK 1 — value_check\n \nFirst, extract the EXPECTED VALUE for each parameter directly from INPUT_TEXT.\n \nexpected_value MUST:\n- Be explicitly grounded in INPUT_TEXT.\n- Never be inferred.\n- Never be null.\n- Reflect normalized enum mapping if applicable.\n \nNORMALIZATION RULE (HIGH PRIORITY):\n \nIf EXTRACTED_PARAMETERS contains normalized classifications,\nthey are valid IF they are grounded in INPUT_TEXT.\n \nExamples:\n"normal goods" → Normal\n"frozen fish" → Frozen\n"glassware" → Fragile\n"fresh apples" → Food\n \nNormalization DOES NOT count as hallucination.\n \nvalue_check = Pass\n- If actual_value matches expected_value semantically.\n \nvalue_check = Fail\n- If actual_value contradicts expected_value.\n- If actual_value is not grounded.\n- If actual_value is fabricated.\n \nCRITICAL GATING RULE:\n \nIf value_check = Fail:\n- format_check MUST be "Fail"\n \n------------------------------------------------------------\n \nCHECK 2 — format_check\n \nEvaluate ONLY if value_check = Pass.\n \nUse EXPECTED_PARAMETER_SCHEMA for type validation.\n \nPass:\n- actual_value matches schema type.\n- If schema type contains "|", treat it as a union — Pass if actual_value matches ANY listed type.\n  Example: "number|string" means both 12 and "12kg" are valid format-wise.\n \nFail:\n- Type mismatch with ALL listed types.\n- Malformed structure.\n \nValid types:\n- string\n- number\n \n------------------------------------------------------------\nSTRICT RULES\n------------------------------------------------------------\n \n- Do NOT infer missing values.\n- Do NOT use external knowledge.\n- Respect gating rules EXACTLY.\n- Every schema parameter MUST appear once in output.\n- parameter_name MUST match schema key exactly.\n- expected_value MUST come from INPUT_TEXT.\n- actual_value MUST come from EXTRACTED_PARAMETERS.\n- No extra commentary outside JSON.\n \n------------------------------------------------------------\nOUTPUT FORMAT (STRICT JSON ARRAY)\n------------------------------------------------------------\n \n[\n  {\n    "parameter_name": "string",\n    "expected_parameter_value": "string",\n    "actual_parameter_value": "string",\n    "value_check": "Pass | Fail",\n    "format_check": "Pass | Fail",\n    "reason": "short deterministic explanation"\n  }\n]\n \n------------------------------------------------------------\nFEW SHOT EXAMPLE\n------------------------------------------------------------\n \nEXPECTED_PARAMETER_SCHEMA:\n{"weight":"number|string"}\n \nINPUT_TEXT:\n"Ship 12kg box"\n \nEXTRACTED_PARAMETERS:\n{"weight":"12kg"}\n \nOutput:\n[\n  {\n    "parameter_name":"weight",\n    "expected_parameter_value":"12kg",\n    "actual_parameter_value":"12kg",\n    "value_check":"Pass",\n    "format_check":"Pass",\n    "reason":"Value grounded and matches union type number|string."\n  }\n]\n',
    },
    "summary_accuracy": {
        "claim_segmentation": '\nRole:\nYou are a CLAIM SEGMENTATION ENGINE.\n\nYour task is to extract ATOMIC FACTUAL OUTPUT FRAGMENTS from a summary.\n\nInputs:\nsummary:\n{SUMMARY}\n\nInstructions:\nDefinition:\nAn Output Fragment is a sentence or clause that asserts something factual\nthat could be verified against execution context.\n\nImportant Rules (STRICT):\n- Do NOT infer information.\n- Do NOT rewrite or paraphrase.\n- Preserve the ORIGINAL wording exactly.\n- Exclude questions, suggestions, and meta statements.\n- Exclude sentences that are purely evaluative, emotional, or conversational.\n- DO NOT verify correctness.\n- DO NOT use external knowledge.\n\n------------------------------------------------------------\nCRITICAL SEGMENTATION RULES (NEW — HIGH PRIORITY)\n------------------------------------------------------------\n\nYou MUST preserve semantic dependency.\n\nDO NOT split fragments when:\n\n1) Parenthetical qualifiers exist:\n   Example:\n   "Estimated cost is $100 (normalized cost matches this amount)"\n   MUST remain ONE fragment.\n\n2) Referential phrases exist:\n   "this amount", "that value", "these results", "which indicates"\n   MUST remain attached to the main clause.\n\n3) Dependent explanatory clauses exist:\n   - normalized\n   - adjusted\n   - derived\n   - calculated\n   - mapped\n   - based on\n\n4) If removing part of the sentence causes ambiguity,\n   the fragment MUST remain whole.\n\nSplit ONLY when:\n- Two independent factual assertions exist with no dependency.\n\n------------------------------------------------------------\nDecision Boundary Rules:\n------------------------------------------------------------\n\nA fragment MUST be trace-verifiable.\n\nDo NOT isolate:\n- Parentheses (...)\n- trailing qualifiers\n- explanatory commas\n- subordinate clauses beginning with "which", "where", "that"\n\n------------------------------------------------------------\nOutput Format:\nReturn STRICT JSON only:\n\n{\n  "claims": ["string"]\n}\n\nFew Shot Examples:\n\nExample 1:\nInput Summary:\n"The plan ID is PLAN_1. It is a FULL delivery."\nOutput:\n{\n  "claims": [\n    "The plan ID is PLAN_1",\n    "It is a FULL delivery"\n  ]\n}\n\nExample 2 - Parenthesis MUST NOT split:\nInput:\n"Estimated cost for the move is $1,264,076 (normalized cost matches this amount)"\nOutput:\n{\n  "claims": [\n    "Estimated cost for the move is $1,264,076 (normalized cost matches this amount)"\n  ]\n}\n\nExample 3:\nInput Summary:\n"Weather conditions are sunny with a risk of 0.1, which should not impact shipment."\nOutput:\n{\n  "claims": [\n    "Weather conditions are sunny with a risk of 0.1"\n  ]\n}\n',
        "claim_verification": '\nYou are a FACTUAL CONSISTENCY VERIFIER.\n\nYour task is to determine whether the given SUMMARY OUTPUT FRAGMENT is SUPPORTED by the provided CONTEXT.\n\nDEFINITIONS:\n- A SUMMARY OUTPUT FRAGMENT is a factual statement extracted from the system output.\n- The CONTEXT is the ONLY source of truth.\n\nIMPORTANT RULES:\n1. Use ONLY CONTEXT.\n2. A fragment is SUPPORTED ONLY IF explicit evidence exists.\n3. Supporting evidence MUST be copied verbatim or near-verbatim.\n4. If no explicit evidence exists -> "No Match".\n5. Do NOT infer or reason beyond text.\n\nContext Search Procedure (MANDATORY):\nYou MUST actively search the CONTEXT to locate supporting evidence.\nThe CONTEXT may contain nested structured data (JSON-like objects).\nYou MUST inspect fields, keys, and values inside the structure.\n\nStep-by-step search strategy:\n1. Identify the subject of the fragment (plan, vendor, route, date, cost, etc.).\n2. Scan the CONTEXT for fields whose key names or values relate to that subject.\n3. Verify whether an explicit value matches the fragment exactly.\n4. You MUST always extract RELEVANT CONTEXT SIGNALS when they exist.\n\nRELEVANT CONTEXT SIGNALS include:\n- matching entities\n- matching attributes\n- matching numeric values\n- related fields that partially align with the fragment\n\nIMPORTANT:\nSupporting Context Evidence is NOT limited to full matches.\nEven when the final status is "No Match", you MUST still include\nany relevant context snippets that relate to the fragment.\n\n5. Evidence Match Status logic:\n\nMatch:\n- explicit statement exists confirming the fragment.\n\nNo Match:\n- context contains related facts but lacks explicit confirmation,\n  OR introduces conflicting semantics.\n\nEvidence presence does NOT imply Match.\n\n6. Do NOT assume relationships between fields unless explicitly stated.\n\nEvidence Selection Rules:\n- Prefer exact key-value matches over descriptive text.\n- Supporting statements must come directly from CONTEXT content.\n- Do NOT summarize or paraphrase the evidence.\n\nExplicit Grounding Rules:\n\n- A JSON field is valid support ONLY if:\n  key semantically matches fragment subject AND\n  value matches fragment attribute.\n\n- Date relationships MUST be explicit.\n  "delivery by X" does NOT imply "requested on X".\n\n- NUMERIC MATCHING RULE (UPDATED - STRICT)\n\nYou MAY treat values as equal when differences are caused ONLY by\nformatting or precision normalization.\n\nAllowed equivalence cases:\n\n1) Formatting normalization:\n   "$1,264,076" == 1264076\n\n2) Float vs integer:\n   1264076 == 1264076.0\n\n3) Precision truncation caused by presentation:\n   98888.01 == 98888\n   ONLY when:\n   - the integer portion is identical\n   - difference is <= 1.0\n   - fragment wording suggests rounding or normalization\n\nSTRICTLY FORBIDDEN:\n- Rounding to nearest thousand/hundred.\n- Percentage or arithmetic reasoning.\n- Any comparison where integer portion differs.\n\nOutput Format:\n{\n  "Supporting Context Evidence": "string",\n  "Evidence Match Status": "Match | No Match",\n  "Verification Reason": "string"\n}\n\nFew Shot Examples:\n\nExample 1 - Direct Match\nInput Fragment:\n"The plan ID is PLAN_1"\nContext:\nPLAN_ID = PLAN_1\nOutput:\n{\n  "Supporting Context Evidence": "PLAN_ID = PLAN_1",\n  "Evidence Match Status": "Match",\n  "Verification Reason": "Context explicitly states PLAN_1."\n}\n\nExample 2 - No Match\nFragment:\n"You requested the shipment on 2026-10-01"\nContext:\ndelivery_date = 2026-10-01\nOutput:\n{\n  "Supporting Context Evidence": "",\n  "Evidence Match Status": "No Match",\n  "Verification Reason": "Delivery date is not equivalent to request date."\n}\n\nExample 3 - Structured JSON Match\nFragment:\n"it is a FULL delivery"\nContext:\n{\'delivery_type\': \'full\'}\nOutput:\n{\n  "Supporting Context Evidence": "\'delivery_type\': \'full\'",\n  "Evidence Match Status": "Match",\n  "Verification Reason": "Explicit structured field confirms delivery type."\n}\n',
    },
    "trajectory_match": {
        "audit": '\nRole:\nYou are an Expert Agent-Trajectory Evaluator.\n\nYour task is to determine whether the ACTUAL agent execution trajectory\nconforms to the REFERENCE agent trajectory under a specific matching MODE.\n\nTradeoff Scenarios (IMPORTANT):\nThe following scenarios define acceptable deviations that MUST be treated as\nneutral behaviour during evaluation if encountered:\n{TRADEOFF_TEXT}\n\nYou must behave deterministically.\nYou must not infer intent, repair execution, or reinterpret system behavior.\n\nInputs:\nreference_outputs:\n{REFERENCE_OUTPUTS_PRETTY}\n\nactual_outputs:\n{ACTUAL_OUTPUTS_PRETTY}\n\nmode:\n{MODE}\n\nInstructions:\n\nMeta Instructions:\nReturn ONLY valid JSON matching output_structure_mandate.\nDo NOT include explanations outside the JSON object.\nDo NOT infer missing agents, intent, or system behavior.\n\nGlobal Context:\nThis evaluation is part of a MULTI-MODE agent trajectory analysis.\nThe same reference_outputs and actual_outputs may be evaluated across multiple modes.\nYour per-mode decision MUST remain logically consistent across modes.\n\nCRITICAL MODE EXCLUSIVITY RULES (MANDATORY):\n\n- If strict mode = Pass → ALL other modes MUST return Fail.\n- subset mode requires: len(actual_outputs) < len(reference_outputs)\n- superset mode requires: len(actual_outputs) > len(reference_outputs)\n- unordered mode requires:\n  same agent names BUT execution order must differ.\n- Modes must NEVER overlap.\n\nDefinitions:\n\nAgent Invocation:\nEvery occurrence of an agent in the sequence is significant.\nDuplicate executions MUST be preserved and compared.\nDo NOT collapse repeated agents into a single occurrence.\n\nReference Outputs:\nThe authoritative, expected agent sequence.\n\nActual Outputs:\nThe observed agent invocation sequence from execution.\n\nMode Semantics:\n\nstrict:\nactual_outputs MUST exactly match reference_outputs.\n\nunordered:\nSame agents but order different.\n\nsubset:\nActual must be smaller than expected.\n\nsuperset:\nActual must be larger than expected.\n\nEvaluation Rules:\n\nThe reason MUST describe:\n- where order differs\n- which agents are missing\n- which agents appear extra\n- how many times agents appear when relevant\n\nThe reason MUST include a short comparison narrative like:\n"reference has X while actual has Y".\n\nUse simple wording:\nUse "larger", "smaller", "extra", "missing".\nAvoid complex phrases.\n\nAnalysis Approach:\n\n1. Compare sequences.\n2. Check ordering.\n3. Count extra occurrences.\n4. Count missing occurrences.\n5. Produce a detailed but simple explanation.\n\nOutput Format:\n\n{\n  "match": "Pass | Fail",\n  "mode": "{MODE}",\n  "reference_sequence": {REFERENCE_OUTPUTS_INLINE},\n  "actual_sequence": {ACTUAL_OUTPUTS_INLINE},\n  "missing_agents": ["string"],\n  "extra_agents": ["string"],\n  "reason": "Detailed comparison explanation"\n}\n',
        "overall_reason": '\nReturn STRICT JSON only.\n\nRole:\nYou are an evaluation summarizer.\n\nUse SIMPLE wording.\n\nExplain overall trajectory behaviour using all modes.\n\nInputs:\n{PER_MODE_SUMMARY}\n\nOutput:\n{\n  "overall_reason": "string"\n}\n',
    },
    "plan_accuracy": {
        "audit": '\nYou are a STRICT CONTEXT-AWARE PLAN STRUCTURE AUDITOR.\n\nYour responsibility is to audit a GENERATED EXECUTION PLAN\nagainst SUPERVISOR WORKFLOW RULES and STRUCTURAL FLOW LOGIC.\n\nYou are NOT evaluating writing quality, grammar, or style.\nYou are performing a STRUCTURAL WORKFLOW AUDIT.\n\nYou MUST behave deterministically and mechanically.\n\n============================================================\nPRIMARY OBJECTIVE\n============================================================\n\nYou MUST perform ALL of the following:\n\n1) SEMANTIC STEP FRAGMENTATION\n2) RULE TEXT MATCHING\n3) RULE ACCURACY VALIDATION\n4) FLOW ACCURACY VALIDATION\n\nThese MUST be done in ONE PASS.\n\n============================================================\nTASK 1 — SEMANTIC STEP FRAGMENTATION (STRICT)\n============================================================\n\nYou MUST extract MEANINGFUL EXECUTION STEPS from generated_plan.\n\nA VALID fragment MUST:\n\n✔ represent a COMPLETE execution action\n✔ contain an AGENT or EXECUTION SUBJECT\n✔ be independently auditable\n\nDO NOT split sentences based on grammar alone.\n\nVALID EXAMPLES:\n- "Step 1: Invoke ContextBuilder with parameters..."\n- "TaskPlanner generates routing combinations"\n\nINVALID FRAGMENTS:\n- "generates routing combinations"\n- "and then sends email"\n- clauses without agent/action\n\n------------------------------------------------------------\nFRAGMENTATION SIGNALS (HIGH PRIORITY)\n------------------------------------------------------------\n\nPrefer splitting using:\n\n- "Step X:"\n- "Task X:"\n- Explicit agent transitions\n- Workflow stage boundaries\n\n============================================================\nTASK 2 — CONTEXT PRESERVATION (CRITICAL)\n============================================================\n\nYou MUST internally carry PREVIOUS steps as background context\nwhen evaluating a fragment.\n\nExample:\n\nStep 1: ContextBuilder gathers data\nStep 2: Forecast computes costs\n\nWhen evaluating Step 2:\n\n✔ consider Step 1 internally\n❌ DO NOT include Step 1 text in Output Fragment\n\nOutput Fragment MUST contain ONLY the CURRENT STEP TEXT.\n\n============================================================\nTASK 3 — RULE MATCHING (STRICT TEXT MATCH)\n============================================================\n\nFor EACH Output Fragment:\n\nYou MUST locate the EXACT matching rule text inside supervisor_rules.\n\nRules:\n\n✔ Copy rule text VERBATIM\n✔ Do NOT paraphrase\n✔ Do NOT synthesize new rules\n✔ Do NOT merge multiple rules\n\nIf NO matching rule exists:\n\nReturn:\n"Matched Rule Fragment": "N/A"\n\n------------------------------------------------------------\nRULE MATCHING STRATEGY\n------------------------------------------------------------\n\nMatch based on:\n\n- Agent name\n- Execution responsibility\n- Action semantics\n\nExample:\n\nFragment:\n"Optimization ranks the plans"\n\nMatching rule:\n"6. **Optimization**: Ranks the plans based on utility..."\n\n============================================================\nTASK 4 — PLAN OUTPUT RULE ACCURACY\n============================================================\n\npass:\n✔ Agent exists in supervisor_rules\n✔ Action allowed by rules\n✔ No forbidden behaviour\n\nfail:\n✖ Agent not present in rules\n✖ Action contradicts rule text\n✖ Required workflow stage skipped\n\nIMPORTANT:\nRule Accuracy MUST be judged ONLY using supervisor_rules.\n\n============================================================\nTASK 5 — PLAN OUTPUT FLOW ACCURACY\n============================================================\n\npass:\n✔ Logical ordering preserved\n✔ Dependencies satisfied\n✔ Workflow progression valid\n\nfail:\n✖ Execution before prerequisite\n✖ Circular logic\n✖ Impossible ordering\n\nFlow Accuracy evaluates INTERNAL PLAN STRUCTURE ONLY.\n\n============================================================\nSTRICT GLOBAL CONSTRAINTS\n============================================================\n\n- DO NOT rewrite fragments\n- DO NOT invent rules\n- DO NOT expose hidden context\n- DO NOT evaluate language quality\n- DO NOT combine fragments\n- DO NOT reference user intent in scoring\n- ALWAYS return JSON only\n\n============================================================\nINPUTS\n============================================================\n\nuser_input:\n{USER_INPUT}\n\nsupervisor_rules:\n{RULES}\n\ngenerated_plan:\n{PLAN}\n\n============================================================\nOUTPUT FORMAT (STRICT JSON)\n============================================================\n\n{\n  "fragments":[\n    {\n      "Output Fragment":"string",\n      "Matched Rule Fragment":"string | N/A",\n      "Plan Output Rule Accuracy":"pass | fail",\n      "Plan Output Flow Accuracy":"pass | fail",\n      "Rule Accuracy Reason":"string",\n      "Flow Accuracy Reason":"string"\n    }\n  ]\n}\n\n============================================================\nFEW SHOT EXAMPLES\n============================================================\n\nExample 1 — Perfect Alignment\n\nSupervisor Rule:\n"3. **TaskPlanner**: Formulates shipment order..."\n\nFragment:\n"Step 3: TaskPlanner formulates shipment order..."\n\nOutput:\n{\n  "Output Fragment":"Step 3: TaskPlanner formulates shipment order...",\n  "Matched Rule Fragment":"3. **TaskPlanner**: Formulates shipment order...",\n  "Plan Output Rule Accuracy":"pass",\n  "Plan Output Flow Accuracy":"pass",\n  "Rule Accuracy Reason":"Fragment directly matches TaskPlanner rule.",\n  "Flow Accuracy Reason":"Correctly follows prior verification stage."\n}\n\n------------------------------------------------------------\n\nExample 2 — Rule Violation\n\nSupervisor Rule:\n"Only Optimization ranks vendors."\n\nFragment:\n"Forecast ranks vendors by score"\n\nOutput:\n{\n  "Output Fragment":"Forecast ranks vendors by score",\n  "Matched Rule Fragment":"N/A",\n  "Plan Output Rule Accuracy":"fail",\n  "Plan Output Flow Accuracy":"pass",\n  "Rule Accuracy Reason":"Forecast is not authorized to rank vendors.",\n  "Flow Accuracy Reason":"Ordering itself is logical."\n}\n\n------------------------------------------------------------\n\nExample 3 — Flow Violation\n\nFragment:\n"Execute sends email before Compliance checks"\n\nOutput:\n{\n  "Output Fragment":"Execute sends email before Compliance checks",\n  "Matched Rule Fragment":"10. **Execute**: Saves the confirmed order to DB and sends an email.",\n  "Plan Output Rule Accuracy":"pass",\n  "Plan Output Flow Accuracy":"fail",\n  "Rule Accuracy Reason":"Execute action allowed by rules.",\n  "Flow Accuracy Reason":"Execution occurs before compliance which violates workflow ordering."\n}\n',
        "overall_reason": '\nYou are a PLAN ACCURACY SUMMARIZER.\n\nExplain structural correctness patterns.\n\nUse SIMPLE wording.\nDescribe ONLY:\n- rule accuracy trends\n- flow accuracy trends\n\nInput:\n{FRAGMENT_RESULTS}\n\nReturn a JSON object with this structure:\n{"overall_reason": "your summary text here"}\n',
    },
}

TOOL_CALL_ACCURACY_SYSTEM_PROMPT = '\n    Role:\nYou are an AUTOMATED TOOL CALL VERIFICATION ENGINE.\nYou are a STRICT, MECHANICAL AUDITOR.\nYou are NOT a planner, assistant, or problem solver.\nYour ONLY responsibility is to verify whether tool calls that were ACTUALLY EXECUTED conform exactly to their authoritative specifications.\nYou must behave deterministically.\nYou must not infer, repair, or reinterpret behavior.\n\nInputs:\n{EXPECTED_TOOLS}: Authoritative specification of valid tools. Defines valid tool names, allowed parameter names per tool, parameter_policy ("exact" or "any_subset"), whether parameter format checking is enforced via "strict", and parameter formats (if defined).\n{ACTUAL_TOOL_CALLS}: The complete set of tool calls that were executed. Defines the ONLY evaluation universe.\n\nInstructions:\nEvaluation Scope (Critical):\nEvaluate ONLY tools listed in ACTUAL_TOOL_CALLS.\nDo NOT invent, infer, or mention tools that were not called.\nDo NOT penalize missing tool calls.\nDo NOT apply external or domain knowledge.\n\nEvaluation Order (Strict and Mandatory):\nYou MUST apply checks in the following order, with NO deviation:\n1. Tool Name Check\n2. Tool Parameter Check\n3. Parameter Format Check\n\nGating Rules (Mandatory Control Flow):\nIf Tool Name Check = incorrect:\n- Tool Parameter Check → skipped\n- Parameter Format Check → skipped\n\nIf Tool Parameter Check = incorrect:\n- Parameter Format Check → skipped\n\nCheck Definitions:\n\n1. Tool Name Check\ncorrect:\n- tool_name exists as a key in EXPECTED_TOOLS\nincorrect:\n- tool_name does NOT exist in EXPECTED_TOOLS\n\n2. Tool Parameter Check\nValidation depends on parameter_policy defined in EXPECTED_TOOLS.\n\nparameter_policy = "exact":\n- Provided parameter keys MUST match EXACTLY\n- No missing parameters\n- No extra parameters\n\nparameter_policy = "any_subset":\n- Provided parameter keys MUST be a subset of allowed_parameters\n- Any extra parameter NOT listed is INVALID\n\nIf Tool Name Check is incorrect, this check MUST be skipped.\n\n3. Parameter Format Check\nValidate ONLY parameter VALUE formats.\nParameter formats are defined INSIDE EXPECTED_TOOLS.\nDo NOT infer or assume formats.\nIf no format is defined for a parameter, do NOT validate it.\n\nIf Tool Parameter Check is incorrect, this check MUST be skipped.\n\nAllowed Judgment Values:\nYou MUST use ONLY:\n"pass"\n"fail"\n\nRules:\n- Any skipped check MUST be marked as "fail".\n- Do NOT output "skipped".\n\n\nReason Construction Rules (STRICT):\n\nThe reason field MUST include the expected schema formats\nfor ALL provided arguments.\n\nFormat requirements:\n\n- Use the word "arguments" (never parameters).\n- When argument format check passes, append the expected formats\n  in brackets using this pattern:\n\n  expected_formats: arg1(type), arg2(type)\n\nExample:\n"Tool recognized; provided arguments comply with exact policy; argument value types align with schema expectations; expected_formats: vendor_id_1(string), vendor_id_2(string)."\n\nRules:\n\n- expected_formats MUST come from EXPECTED_TOOLS.allowed_parameters.\n- Include ONLY arguments that were provided in the ACTUAL_TOOL_CALL.\n- Maintain professional audit tone.\n- Do NOT add recommendations or suggestions.\n\n\nOutput Format:\nReturn ONLY a JSON object with the following structure:\n{\n  "tools": [\n    {\n      "tool_name": "string",\n      "tool_name_check": "pass | fail",\n      "tool_parameter_check": "pass | fail",\n      "parameter_format_check": "pass | fail",\n      "reason": "short, precise justification"\n    }\n  ],\n  "overall_reason": "High-level assessment of tool-call correctness"\n}\n\nFew Shot Examples:\n\nExample 1 — Fully Correct Tool Call\n\nEXPECTED_TOOLS:\n{\n  "analytics_vendors_find": {\n    "allowed_parameters": ["base_location", "limit"],\n    "parameter_policy": "any_subset",\n    "strict": false\n  }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_vendors_find",\n    "parameters": {\n      "base_location": "BOSTON"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_vendors_find",\n      "tool_name_check": "pass",\n      "tool_parameter_check": "pass",\n      "parameter_format_check": "pass",\n      "reason": "Tool recognized; provided arguments comply with subset policy; argument value types align with schema expectations; expected_formats: base_location(string)."\n    }\n  ],\n  "overall_reason": "All executed tools conform to their specifications."\n}\n\nExample 2 — Incorrect Tool Name\n\nEXPECTED_TOOLS:\n{\n  "analytics_vendors_find": { }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_vendor_search",\n    "parameters": {\n      "base_location": "BOSTON"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_vendor_search",\n      "tool_name_check": "fail",\n      "tool_parameter_check": "fail",\n      "parameter_format_check": "fail",\n      "reason": "Tool not recognized; executed tool is not present in EXPECTED_TOOLS, therefore argument validation and format evaluation cannot be performed."\n    }\n  ],\n  "overall_reason": "At least one executed tool is not defined in EXPECTED_TOOLS."\n}\n\nExample 3 — Argument Mismatch (Exact Policy)\n\nEXPECTED_TOOLS:\n{\n  "analytics_confirmed_order_lookup": {\n    "allowed_parameters": ["order_id"],\n    "parameter_policy": "exact",\n    "strict": false\n  }\n}\n\nACTUAL_TOOL_CALLS:\n[\n  {\n    "tool_name": "analytics_confirmed_order_lookup",\n    "parameters": {\n      "order_id": "123",\n      "status": "confirmed"\n    }\n  }\n]\n\nOutput:\n{\n  "tools": [\n    {\n      "tool_name": "analytics_confirmed_order_lookup",\n      "tool_name_check": "pass",\n      "tool_parameter_check": "fail",\n      "parameter_format_check": "fail",\n      "reason": "Tool recognized; provided arguments violate exact policy due to unexpected argument \'status\'; expected_formats: order_id(string)."\n    }\n  ],\n  "overall_reason": "One or more executed tools violate argument policy constraints."\n}\n    '


# ---- Builder functions copied verbatim from prompts.py ----
def build_parameter_extraction_overall_reason_prompt(audit_results):
    return f'''
Role:
You are a STRICT Evaluation Summary Generator.

Purpose:
Produce a concise structural summary describing grounding behaviour.

CRITICAL:
- Do NOT provide suggestions.
- Do NOT mention format/type issues.
- Describe only grounding (value_check).
- Return ONLY raw JSON. No markdown, no code fences.

audit_results:
{json.dumps(audit_results, indent=2)}

Return ONLY this JSON object (no markdown, no extra text):

{{
  "overall_reason": "1-3 concise factual sentences."
}}
'''


def build_tool_call_accuracy_user_prompt(expected_tools, slimmed_tool_calls):
    return f"""
EXPECTED TOOLS:
{json.dumps(expected_tools, indent=2)}

ACTUAL TOOL CALLS:
{json.dumps(slimmed_tool_calls, indent=2)}
"""


def build_goal_accomplishment_prompt(task, actual_outcome):
    return f'''
meta_instruction:
You MUST strictly follow the analysis_approach and output_structure_mandate.
Produce ONLY the required JSON object, without commentary outside the JSON.

metric_name:
Goal Accomplishment — Execute Plan Summary

role:
You are a Goal Accomplishment Evaluator.
Your responsibility is to determine whether the Execute Plan Summary
directly answers the user's request.

goal:
Evaluate whether each explicit fragment of the user's query
is addressed in the Execute Plan Summary.

inputs:

task:
{task}

actual_outcome:
{actual_outcome}

evaluation_definition:

User Query Fragment:
A concrete requirement, constraint, or intent expressed by the user.
Examples include:
- item details
- delivery conditions
- routing constraints
- optimisation requests
- confirmations or decisions

Match:
A clear corresponding statement exists in actual_outcome that fulfills
or directly addresses the fragment.

No Match:
No clear statement exists that addresses the fragment.

analysis_approach:

STEP 1:
Break the task into explicit query fragments.
Fragments must be grounded only in provided text.

STEP 2:
Search actual_outcome for statements that address each fragment.

STEP 3:
For each fragment:
- mark Match if a direct or semantically equivalent statement exists.
- mark No Match if absent.

STEP 4:
Assess overall accomplishment:
- If most critical fragments are Match → higher verdict.
- If several fragments are Missed → lower verdict.

STEP 5:
Provide a concise reasoning summary describing overall alignment.

strict_rules:

- Evaluate ONLY provided inputs.
- Do NOT infer hidden intent.
- Do NOT add fragments not present in task.
- Do NOT use external knowledge.
- Do NOT judge writing quality or style.

output_structure_mandate:

Return ONLY JSON:

{{
  "verdict": number,
  "reason": "1-3 sentences summarizing overall goal accomplishment",
  "fragments": [
    {{
      "query_fragment": "string",
      "summary_evidence": "string",
      "status": "Match | No Match"
    }}
  ]
}}

few_shot_example:

task:
Ship frozen salmon using airways only and deliver before Feb 5.

actual_outcome:
Plan uses air freight via UPS. Delivery ETA Feb 4.

Output:
{{
  "verdict": 1.0,
  "reason": "All user requirements appear in the summary.",
  "fragments": [
    {{
      "query_fragment": "frozen salmon shipment",
      "summary_evidence": "Plan uses air freight via UPS",
      "status": "Match"
    }},
    {{
      "query_fragment": "airways only",
      "summary_evidence": "air freight via UPS",
      "status": "Match"
    }},
    {{
      "query_fragment": "deliver before Feb 5",
      "summary_evidence": "Delivery ETA Feb 4",
      "status": "Match"
    }}
  ]
}}
'''.strip()


print(f"Loaded native prompts for {len(NATIVE_PROMPTS)} registry metrics "
      f"+ tool_call system prompt + 3 builder functions.")


## 5. Expected Trajectory and Tool Calls

Defines the only external reference artifacts used during evaluation. `trajectory_match` compares the expected and actual stage sequence; `tool_call_accuracy` compares expected operations with actual calls. The other metrics compare artifacts already present in the trace.


In [ ]:
# ============================================================
# Expected Trajectory and Expected Tool Calls
# ============================================================

EXPECTED_TRAJECTORY = [
    "Parameter Extraction Agent",
    "Supervisor: Plan & Route",
    "Research Agent",
    "Web Search Sub-Agent",
    "Document Retrieval Sub-Agent",
    "Source Validator Sub-Agent",
    "Supervisor: Review Research",
    "Analysis Agent",
    "Market Sizing Sub-Agent",
    "Competitor Benchmarking Sub-Agent",
    "Supervisor: Review Analysis",
    "Research Agent",
    "Web Search Sub-Agent",
    "Supervisor: Review Addendum",
    "Report Writer Agent",
    "Drafting Sub-Agent",
    "Formatting & Citation Sub-Agent",
    "Supervisor: Final Review",
]

EXPECTED_TOOL_CALLS = {
    "Research Agent": [
        "Web Search Sub-Agent",
        "Document Retrieval Sub-Agent",
        "Source Validator Sub-Agent",
    ],
    "Analysis Agent": [
        "Market Sizing Sub-Agent",
        "Competitor Benchmarking Sub-Agent",
    ],
    "Report Writer Agent": [
        "Drafting Sub-Agent",
        "Formatting & Citation Sub-Agent",
    ],
}

EXPECTED_TOOL_NAMES = {
    tool_name
    for tools in EXPECTED_TOOL_CALLS.values()
    for tool_name in tools
}

print("Expected trajectory and expected tool-call mapping loaded.")


Expected trajectory and expected tool-call mapping loaded.


## 6. Selected Metrics

Lists the six metrics executed for every retrieved trace.


In [ ]:
# ============================================================
# Metric Selection
# ============================================================

SELECTED_METRICS = [
    "trajectory_match",
    "parameter_extraction",
    "plan_accuracy",
    "tool_call_accuracy",
    "summary_accuracy",
    "goal_accomplishment",
]

print("Selected evaluation metrics:")
for index, metric in enumerate(SELECTED_METRICS, start=1):
    print(f"{index}. {metric}")


Selected evaluation metrics:
1. trajectory_match
2. parameter_extraction
3. plan_accuracy
4. tool_call_accuracy
5. summary_accuracy
6. goal_accomplishment


## 7. Retrieve and Reconstruct Recent Traces

Set `LOOKBACK_HOURS` to select all root traces started in that period. The cell retrieves them from LangSmith and reconstructs each complete parent-child trace hierarchy.


In [ ]:
# ============================================================
# Retrieve LangSmith Traces in a Time Window
# ============================================================

LOOKBACK_HOURS = 1

if LOOKBACK_HOURS <= 0:
    raise ValueError("LOOKBACK_HOURS must be greater than zero.")

window_end = datetime.now(timezone.utc)
window_start = window_end - timedelta(hours=LOOKBACK_HOURS)

root_runs = list(
    langsmith_client.list_runs(
        project_name=LANGSMITH_PROJECT,
        is_root=True,
        start_time=window_start,
    )
)

# Make the evaluation order deterministic and easy to read.
root_runs.sort(key=lambda run: run.start_time or window_start)

if not root_runs:
    raise RuntimeError(
        f"No root traces found in project '{LANGSMITH_PROJECT}' "
        f"during the last {LOOKBACK_HOURS} hour(s)."
    )

print("=" * 70)
print("LangSmith Traces Loaded")
print("=" * 70)
print(f"Project          : {LANGSMITH_PROJECT}")
print(f"Lookback window  : last {LOOKBACK_HOURS} hour(s)")
print(f"Window start UTC : {window_start.isoformat()}")
print(f"Window end UTC   : {window_end.isoformat()}")
print(f"Root traces      : {len(root_runs)}")

for index, run in enumerate(root_runs, start=1):
    print(f"{index}. {run.name} | {run.id} | {run.start_time}")

print("=" * 70)


def build_trace_tree(run):
    """Recursively reconstruct one LangSmith execution tree."""
    children = list(langsmith_client.list_runs(parent_run_id=run.id))
    children.sort(key=lambda child: child.start_time or run.start_time)

    return {
        "id": str(run.id),
        "name": run.name,
        "run_type": run.run_type,
        "inputs": run.inputs or {},
        "outputs": run.outputs or {},
        "metadata": run.extra or {},
        "start_time": run.start_time,
        "end_time": run.end_time,
        "children": [build_trace_tree(child) for child in children],
    }


execution_traces = [build_trace_tree(root_run) for root_run in root_runs]

print(f"Reconstructed {len(execution_traces)} trace hierarchy/hierarchies.")


LangSmith Traces Loaded
Project          : multi-agent-demo
Lookback window  : last 1 hour(s)
Window start UTC : 2026-07-16T03:45:09.682830+00:00
Window end UTC   : 2026-07-16T04:45:09.682830+00:00
Root traces      : 5
1. Supervisor Agent | 019f691b-6b77-7370-84f9-7c60fb1086cf | 2026-07-16 04:07:12.503310+00:00
2. Supervisor Agent — variant_A_poor_parameter_and_plan | 019f691b-6ddd-7bc3-8b6e-9ec2b713c6ef | 2026-07-16 04:07:13.117874+00:00
3. Supervisor Agent — variant_B_poor_trajectory_and_tools | 019f691b-6ee2-7802-a3a7-ea877d9592cf | 2026-07-16 04:07:13.378450+00:00
4. Supervisor Agent — variant_C_poor_summary_accuracy | 019f691b-6fd3-7ab0-af61-3ef0c7ff52f5 | 2026-07-16 04:07:13.619806+00:00
5. Supervisor Agent — variant_D_poor_goal_accomplishment | 019f691b-70cc-77a0-b930-343d25b834ea | 2026-07-16 04:07:13.868627+00:00
Reconstructed 5 trace hierarchy/hierarchies.


## 8. Native Multi-Stage Evaluation Working

Reproduces each metric's original working instead of a single judge call: parameter/plan run an audit then an overall-reason summary; trajectory runs the audit once per mode (strict/unordered/subset/superset) then summarizes; summary segments claims then verifies each against the trace; tool-call runs one gated system+user audit; goal-accomplishment runs one fragment-level audit. Every metric keeps its per-fragment output as `traceback` and derives a 1-10 `score` so the downstream report, Excel, and feedback cells are unchanged.

In [ ]:
# ============================================================
# Native Multi-Stage Evaluation Working (ported from prompts.py)
# ============================================================
# Original per-metric flow reproduced faithfully:
#   parameter_extraction : audit -> per-parameter value_check/format_check gate -> overall_reason
#   trajectory_match     : audit run once per MODE (strict/unordered/subset/superset) -> overall_reason
#   plan_accuracy        : audit -> per-step Rule/Flow pass|fail -> overall_reason
#   tool_call_accuracy   : single system+user audit -> per-tool 3 gated checks
#   summary_accuracy     : claim_segmentation -> per-claim claim_verification (Match/No Match)
#   goal_accomplishment  : single audit -> per-fragment Match/No Match + numeric verdict
#
# Each metric keeps its rich per-fragment output as `traceback`, and a derived
# 1-10 `score` (from the pass ratio / verdict) so the existing report, Excel,
# and LangSmith-feedback cells continue to work unchanged.
#
# NOTE on grounding gaps in the demo traces:
#   * plan_accuracy needs `supervisor_rules` and tool_call_accuracy needs a tool
#     schema. The synthetic traces carry neither, so supervisor_rules is
#     synthesized from EXPECTED_TRAJECTORY/EXPECTED_TOOL_CALLS and each expected
#     tool's allowed_parameters is grounded in the params actually observed. The
#     tool-NAME check (is the called tool expected?) is the meaningful signal.

from collections import defaultdict

PARAMETER_SCHEMA = {
    "company_profile": "object",
    "market": "object",
    "target_year": "number",
    "objective": "string",
    "deliverables": "array",
}

TRAJECTORY_MODES = ["strict", "unordered", "subset", "superset"]


# ------------------------------------------------------------
# LLM call + tolerant JSON parsing
# ------------------------------------------------------------
def invoke_llm(system_prompt, user_prompt, force_json_object=True):
    """Call Azure OpenAI. force_json_object=False for prompts returning a top-level array."""
    kwargs = dict(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    if force_json_object:
        kwargs["response_format"] = {"type": "json_object"}
    return azure_client.chat.completions.create(**kwargs).choices[0].message.content


def parse_json(response):
    """Parse a JSON object OR array, tolerating code fences / surrounding text."""
    txt = (response or "").strip()
    if txt.startswith("```"):
        txt = txt.split("\n", 1)[1] if "\n" in txt else ""
        if txt.rstrip().endswith("```"):
            txt = txt.rstrip()[:-3]
    starts = [i for i in (txt.find("{"), txt.find("[")) if i >= 0]
    if not starts:
        raise ValueError("No JSON object or array found in model response.")
    obj, _ = json.JSONDecoder().raw_decode(txt[min(starts):])
    return obj


def _score_from_ratio(passed, total):
    """Map a pass ratio to a 1-10 integer score (None when nothing to score)."""
    if not total:
        return None
    return max(1, min(10, round(10 * passed / total)))


def _missing(metric, msg):
    return {"metric": metric, "score": None, "max_score": 10,
            "reasoning": msg, "traceback": []}


# ------------------------------------------------------------
# Trace preprocessing (unchanged in spirit) + helpers
# ------------------------------------------------------------
def preprocess_execution_trace(execution_trace):
    flattened, agent_spans, tool_spans = [], [], []
    span_lookup = defaultdict(list)

    def visit(node):
        flattened.append(node)
        span_lookup[node["name"]].append(node)
        if node["run_type"] in ("tool", "retriever") or node["name"] in EXPECTED_TOOL_NAMES:
            tool_spans.append(node)
        elif node["run_type"] in ("llm", "chain"):
            agent_spans.append(node)
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace)
    return {"flattened_trace": flattened, "agent_spans": agent_spans,
            "tool_spans": tool_spans, "span_lookup": span_lookup}


def first_span(span_lookup, name):
    matches = span_lookup.get(name, [])
    return matches[0] if matches else None


def span_inputs(span):
    return (span or {}).get("inputs", {}) or {}


def span_outputs(span):
    return (span or {}).get("outputs", {}) or {}


def ordered_trajectory_names(execution_trace):
    """Ordered agent-name sequence (root wrapper excluded)."""
    names = []

    def visit(node, is_root=False):
        if not is_root:
            names.append(node["name"])
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace, is_root=True)
    return names


def collect_evidence_context(span):
    """Upstream evidence a summary should be verified against: the span's own
    inputs plus every descendant (sub-agent / tool) output. Excludes the span's
    OWN summary output, which would otherwise make verification circular."""
    evidence = {"inputs": span_inputs(span), "evidence": {}}

    def gather(node):
        for child in node.get("children", []):
            evidence["evidence"][child["name"]] = span_outputs(child)
            gather(child)

    gather(span)
    return json.dumps(evidence, default=str)


def build_supervisor_rules():
    """Synthesize the authoritative workflow rulebook plan_accuracy expects."""
    lines = ["Supervisor Workflow Rules (authoritative execution order):"]
    for i, stage in enumerate(EXPECTED_TRAJECTORY, 1):
        lines.append(f"{i}. **{stage}**: authorized workflow stage; must run in this order.")
    lines.append("")
    lines.append("Tool-invocation authority:")
    for agent, tools in EXPECTED_TOOL_CALLS.items():
        lines.append(f"- **{agent}** may invoke: {', '.join(tools)}.")
    return "\n".join(lines)


def build_expected_tools_spec(tool_spans):
    """EXPECTED_TOOLS spec for the tool_call prompt (allowed_parameters grounded in observed calls)."""
    observed = defaultdict(set)
    for s in tool_spans:
        observed[s["name"]].update(span_inputs(s).keys())
    return {
        name: {
            "allowed_parameters": sorted(observed.get(name, [])),
            "parameter_policy": "any_subset",
            "strict": False,
        }
        for name in EXPECTED_TOOL_NAMES
    }


# ------------------------------------------------------------
# Per-metric runners (native working)
# ------------------------------------------------------------
def run_parameter_extraction(span):
    if not span:
        return _missing("parameter_extraction", "Parameter Extraction Agent span not found.")
    outputs = span_outputs(span)
    extracted = outputs.get("parameters", outputs)
    audit_prompt = (
        NATIVE_PROMPTS["parameter_extraction"]["audit"]
        .replace("{INPUT_TEXT}", json.dumps(span_inputs(span), indent=2, default=str))
        .replace("{EXTRACTED_PARAMETERS}", json.dumps(extracted, indent=2, default=str))
        .replace("{EXPECTED_PARAMETER_SCHEMA}", json.dumps(PARAMETER_SCHEMA))
    )
    raw = invoke_llm(audit_prompt, "Return the strict JSON array now.", force_json_object=False)
    audit = parse_json(raw)
    if isinstance(audit, dict):
        audit = [audit]
    # Score on value_check (grounding). The native prompt's overall_reason
    # summarizes grounding only, and format_check recognizes just string/number,
    # which cannot validate the nested object/array params in this domain.
    # format_check is retained in the traceback for inspection.
    grounded = sum(1 for r in audit if r.get("value_check") == "Pass")
    score = _score_from_ratio(grounded, len(audit))
    reasoning = ""
    try:
        reason_raw = invoke_llm(build_parameter_extraction_overall_reason_prompt(audit),
                                "Return the JSON object now.")
        reasoning = parse_json(reason_raw).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    return {"metric": "parameter_extraction", "score": score, "max_score": 10,
            "reasoning": reasoning, "traceback": audit, "raw_evaluation": raw}


def run_trajectory_match(expected_names, execution_trace):
    actual_names = ordered_trajectory_names(execution_trace)
    ref_pretty, ref_inline = json.dumps(expected_names, indent=2), json.dumps(expected_names)
    act_pretty, act_inline = json.dumps(actual_names, indent=2), json.dumps(actual_names)

    per_mode = []
    for mode in TRAJECTORY_MODES:
        ap = (
            NATIVE_PROMPTS["trajectory_match"]["audit"]
            .replace("{TRADEOFF_TEXT}", "None. No acceptable deviations are defined; evaluate literally.")
            .replace("{REFERENCE_OUTPUTS_PRETTY}", ref_pretty)
            .replace("{ACTUAL_OUTPUTS_PRETTY}", act_pretty)
            .replace("{MODE}", mode)
            .replace("{REFERENCE_OUTPUTS_INLINE}", ref_inline)
            .replace("{ACTUAL_OUTPUTS_INLINE}", act_inline)
        )
        try:
            res = parse_json(invoke_llm(ap, f"Evaluate strictly under mode='{mode}'. Return JSON now."))
        except Exception as e:
            res = {"mode": mode, "match": "Fail", "reason": f"parse error: {e}"}
        res.setdefault("mode", mode)
        per_mode.append(res)

    strict = next((m for m in per_mode if m.get("mode") == "strict"), {})
    missing = len(strict.get("missing_agents") or [])
    extra = len(strict.get("extra_agents") or [])
    denom = len(expected_names) + len(actual_names)
    if strict.get("match") == "Pass":
        score = 10
    elif (missing + extra) == 0:
        # same agent multiset, order differs (or no diff detail) -> partial credit
        score = 6
    else:
        score = _score_from_ratio(max(0, denom - (missing + extra)), denom) or 1

    reasoning = ""
    try:
        summary = [{"mode": m.get("mode"), "match": m.get("match"), "reason": m.get("reason")}
                   for m in per_mode]
        rr = invoke_llm(
            NATIVE_PROMPTS["trajectory_match"]["overall_reason"]
            .replace("{PER_MODE_SUMMARY}", json.dumps(summary, indent=2)),
            "Return the JSON object now.",
        )
        reasoning = parse_json(rr).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    return {"metric": "trajectory_match", "score": score, "max_score": 10,
            "reasoning": reasoning, "traceback": per_mode,
            "raw_evaluation": json.dumps(per_mode, default=str)[:4000]}


def run_plan_accuracy(span, execution_trace):
    if not span:
        return _missing("plan_accuracy", "Supervisor: Plan & Route span not found.")
    ap = (
        NATIVE_PROMPTS["plan_accuracy"]["audit"]
        .replace("{USER_INPUT}", json.dumps(execution_trace.get("inputs", {}), indent=2, default=str))
        .replace("{RULES}", build_supervisor_rules())
        .replace("{PLAN}", json.dumps(span_outputs(span), indent=2, default=str))
    )
    raw = invoke_llm(ap, "Return the strict JSON now.")
    parsed = parse_json(raw)
    frags = parsed.get("fragments", []) if isinstance(parsed, dict) else parsed
    passed = sum(1 for f in frags
                 if f.get("Plan Output Rule Accuracy") == "pass"
                 and f.get("Plan Output Flow Accuracy") == "pass")
    score = _score_from_ratio(passed, len(frags))
    reasoning = ""
    try:
        rr = invoke_llm(
            NATIVE_PROMPTS["plan_accuracy"]["overall_reason"]
            .replace("{FRAGMENT_RESULTS}", json.dumps(frags, indent=2, default=str)),
            "Return the JSON object now.",
        )
        reasoning = parse_json(rr).get("overall_reason", "")
    except Exception as e:
        reasoning = f"(overall_reason stage failed: {e})"
    return {"metric": "plan_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning, "traceback": frags, "raw_evaluation": raw}


def run_tool_call_accuracy(expected_tools, tool_spans):
    if not tool_spans:
        return _missing("tool_call_accuracy", "No tool/sub-agent spans found in trace.")
    actual = [{"tool_name": s["name"], "parameters": span_inputs(s)} for s in tool_spans]
    raw = invoke_llm(TOOL_CALL_ACCURACY_SYSTEM_PROMPT,
                     build_tool_call_accuracy_user_prompt(expected_tools, actual))
    parsed = parse_json(raw)
    tools = parsed.get("tools", []) if isinstance(parsed, dict) else parsed
    passed = sum(1 for t in tools
                 if t.get("tool_name_check") == "pass"
                 and t.get("tool_parameter_check") == "pass"
                 and t.get("parameter_format_check") == "pass")
    score = _score_from_ratio(passed, len(tools))
    reasoning = parsed.get("overall_reason", "") if isinstance(parsed, dict) else ""
    return {"metric": "tool_call_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning, "traceback": tools, "raw_evaluation": raw}


def run_summary_accuracy(spans):
    if not spans:
        return _missing("summary_accuracy", "No summary-producing spans found.")
    claim_records, total, matched = [], 0, 0
    for span in spans:
        summary_text = json.dumps(span_outputs(span), default=str)
        try:
            claims = parse_json(invoke_llm(
                NATIVE_PROMPTS["summary_accuracy"]["claim_segmentation"].replace("{SUMMARY}", summary_text),
                "Return the JSON object now.",
            )).get("claims", [])
        except Exception as e:
            claim_records.append({"agent": span["name"], "claim": "(segmentation failed)",
                                  "status": "No Match", "evidence": "", "reason": str(e)})
            continue
        context = collect_evidence_context(span)
        for claim in claims:
            try:
                v = parse_json(invoke_llm(
                    NATIVE_PROMPTS["summary_accuracy"]["claim_verification"],
                    json.dumps({"Output Fragment": claim, "context": context}),
                ))
                status = v.get("Evidence Match Status", "No Match")
                evidence = v.get("Supporting Context Evidence", "")
                reason = v.get("Verification Reason", "")
            except Exception as e:
                status, evidence, reason = "No Match", "", f"verify error: {e}"
            total += 1
            matched += 1 if status == "Match" else 0
            claim_records.append({"agent": span["name"], "claim": claim, "status": status,
                                  "evidence": evidence, "reason": reason})
    score = _score_from_ratio(matched, total)
    reasoning = (f"{matched}/{total} summary claims supported by trace context."
                 if total else "No verifiable factual claims were segmented.")
    return {"metric": "summary_accuracy", "score": score, "max_score": 10,
            "reasoning": reasoning, "traceback": claim_records,
            "raw_evaluation": json.dumps(claim_records, default=str)[:4000]}


def run_goal_accomplishment(execution_trace):
    raw = invoke_llm(
        build_goal_accomplishment_prompt(
            json.dumps(execution_trace.get("inputs", {}), default=str),
            json.dumps(execution_trace.get("outputs", {}), default=str),
        ),
        "Return the JSON object now.",
    )
    parsed = parse_json(raw)
    try:
        score = max(1, min(10, round(float(parsed.get("verdict", 0)) * 10)))
    except (TypeError, ValueError):
        score = None
    return {"metric": "goal_accomplishment", "score": score, "max_score": 10,
            "reasoning": parsed.get("reason", ""), "traceback": parsed.get("fragments", []),
            "raw_evaluation": raw}


# ------------------------------------------------------------
# Metric targets + dispatcher (same interface as before)
# ------------------------------------------------------------
def build_metric_targets(execution_trace, trace_data):
    sl = trace_data["span_lookup"]
    return {
        "trajectory_match": {"expected": EXPECTED_TRAJECTORY, "actual": execution_trace},
        "parameter_extraction": {"actual": first_span(sl, "Parameter Extraction Agent")},
        "plan_accuracy": {"actual": first_span(sl, "Supervisor: Plan & Route")},
        "tool_call_accuracy": {"expected": build_expected_tools_spec(trace_data["tool_spans"]),
                               "actual": trace_data["tool_spans"]},
        "summary_accuracy": {"actual": (sl.get("Research Agent", [])
                                        + sl.get("Analysis Agent", [])
                                        + sl.get("Report Writer Agent", []))},
        "goal_accomplishment": {"actual": first_span(sl, "Supervisor: Final Review")},
    }


def evaluate_metric(metric_name, metric_targets, execution_trace):
    t = metric_targets[metric_name]
    try:
        if metric_name == "parameter_extraction":
            return run_parameter_extraction(t["actual"])
        if metric_name == "trajectory_match":
            return run_trajectory_match(t["expected"], t["actual"])
        if metric_name == "plan_accuracy":
            return run_plan_accuracy(t["actual"], execution_trace)
        if metric_name == "tool_call_accuracy":
            return run_tool_call_accuracy(t["expected"], t["actual"])
        if metric_name == "summary_accuracy":
            return run_summary_accuracy(t["actual"])
        if metric_name == "goal_accomplishment":
            return run_goal_accomplishment(execution_trace)
    except (ValueError, json.JSONDecodeError) as error:
        return {"metric": metric_name, "score": None, "max_score": 10,
                "reasoning": f"Evaluator response invalid: {error}",
                "traceback": [], "parse_error": str(error)}
    raise ValueError(f"Unsupported metric: {metric_name}")


print("Native multi-stage evaluation working loaded for all six metrics.")


## 9. Evaluate and Export Results

Runs every selected metric for every retrieved trace. Results are tagged with trace and variant metadata, then written to `evaluation_results.json` and `evaluation_results.xlsx`. The final output prints an average score per trace.


In [ ]:
# ============================================================
# Execute Evaluation Pipeline for All Retrieved Traces
# ============================================================

def get_variant_id(trace_name):
    """Map the logger's root-run name to an expected-result variant ID."""
    prefix = "Supervisor Agent — "
    return trace_name.removeprefix(prefix) if trace_name.startswith(prefix) else "baseline"


evaluation_results = []

print("=" * 70)
print("Running Evaluation Pipeline")
print("=" * 70)

for trace_number, (root_run, execution_trace) in enumerate(
    zip(root_runs, execution_traces), start=1
):
    print(f"\nTrace {trace_number}/{len(root_runs)}: {root_run.name} ({root_run.id})")

    trace_data = preprocess_execution_trace(execution_trace)
    metric_targets = build_metric_targets(execution_trace, trace_data)

    for metric in SELECTED_METRICS:
        print(f"  Evaluating: {metric}")
        try:
            result = evaluate_metric(metric, metric_targets, execution_trace)
        except Exception as error:
            result = {
                "metric": metric,
                "score": None,
                "max_score": 10,
                "reasoning": f"Evaluation failed: {error}",
            }

        result.update({
            "variant_id": get_variant_id(root_run.name),
            "trace_id": str(root_run.id),
            "trace_name": root_run.name,
            "trace_start_time": (
                root_run.start_time.isoformat() if root_run.start_time else None
            ),
            "lookback_hours": LOOKBACK_HOURS,
        })
        evaluation_results.append(result)

print("=" * 70)
print("Evaluation Completed")
print("=" * 70)
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")


# ============================================================
# Process Evaluation Results
# ============================================================

processed_results = [
    {
        "Variant": result.get("variant_id"),
        "Trace ID": result.get("trace_id"),
        "Trace Name": result.get("trace_name"),
        "Trace Start Time": result.get("trace_start_time"),
        "Metric": result.get("metric"),
        "Score": result.get("score"),
        "Max Score": result.get("max_score"),
        "Reasoning": result.get("reasoning"),
        "Recommendation": result.get("recommendation"),
    }
    for result in evaluation_results
]

print(f"Processed {len(processed_results)} metric result(s).")


# ============================================================
# Export Results to JSON
# ============================================================

JSON_OUTPUT_FILE = "evaluation_results.json"

with open(JSON_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=4, ensure_ascii=False)

print(f"✓ JSON results exported to '{JSON_OUTPUT_FILE}'")

# ============================================================
# Export Results to Excel
# ============================================================

EXCEL_OUTPUT_FILE = "evaluation_results.xlsx"

df = pd.DataFrame(processed_results)
df.to_excel(EXCEL_OUTPUT_FILE, index=False)

print(f"✓ Excel results exported to '{EXCEL_OUTPUT_FILE}'")


# ============================================================
# Evaluation Summary
# ============================================================

print("=" * 70)
print("LangSmith Trace Evaluation Completed")
print("=" * 70)
print(f"Lookback window   : {LOOKBACK_HOURS} hour(s)")
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")
print(f"JSON Report       : {JSON_OUTPUT_FILE}")
print(f"Excel Report      : {EXCEL_OUTPUT_FILE}")
print("=" * 70)

summary_df = pd.DataFrame(evaluation_results)
summary_df["score"] = pd.to_numeric(summary_df["score"], errors="coerce")

print("\nAverage score by trace:")
for trace_name, group in summary_df.groupby("trace_name", dropna=False):
    average = group["score"].mean()
    score_text = f"{average:.2f}/10" if pd.notna(average) else "N/A"
    print(f"- {trace_name}: {score_text}")

print("\nEvaluation completed successfully.")


Running Evaluation Pipeline

Trace 1/5: Supervisor Agent (019f691b-6b77-7370-84f9-7c60fb1086cf)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 2/5: Supervisor Agent — variant_A_poor_parameter_and_plan (019f691b-6ddd-7bc3-8b6e-9ec2b713c6ef)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 3/5: Supervisor Agent — variant_B_poor_trajectory_and_tools (019f691b-6ee2-7802-a3a7-ea877d9592cf)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 4/5: Supervisor Agent — variant_C_poor_summary_accuracy (019f691b-6fd3-7ab0-af61-3ef0c7ff52f5)
  Evaluating: tr

## 10. Interactive Metric Details Report

Creates a standalone HTML report after evaluation. Click any metric score to open its metric-level audit evidence: parameter checks, stage comparisons, plan fragments, tool checks, summary claim evidence, or goal-requirement matches.


In [ ]:
# ============================================================
# Interactive Metric Details Report (Pivoted: 1 Trace = 1 Row)
# ============================================================

from html import escape
from pathlib import Path
from collections import defaultdict
import json


def render_detail_table(items):
    """Render rich traceback details nicely."""
    if not items:
        return "<p class='empty'>No structured details available.</p>"

    if isinstance(items, dict):
        items = [items]

    columns = []
    for item in items:
        if isinstance(item, dict):
            for k in item:
                if k not in columns:
                    columns.append(k)

    if not columns:
        return f"<pre>{escape(json.dumps(items, indent=2, ensure_ascii=False, default=str))}</pre>"

    header = "".join(f"<th>{escape(str(k).replace('_', ' ').title())}</th>" for k in columns)
    rows = []
    for item in items:
        if not isinstance(item, dict):
            item = {"value": str(item)}
        cells = []
        for col in columns:
            val = item.get(col, "—")
            if isinstance(val, (dict, list)):
                val = json.dumps(val, ensure_ascii=False, default=str, indent=2)
            cells.append(f"<td><pre style='white-space:pre-wrap;margin:0'>{escape(str(val))}</pre></td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")

    return f"""<div class="detail-scroll"><table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>"""


def build_interactive_report(results, output_file="evaluation_details_report.html"):
    """Pivoted report: One row per trace, columns for each metric score."""
    traces = defaultdict(list)
    for r in results:
        key = (r.get("variant_id", "baseline"), r.get("trace_name", "—"), r.get("trace_id", ""))
        traces[key].append(r)

    metric_order = [
        "trajectory_match",
        "parameter_extraction",
        "plan_accuracy",
        "tool_call_accuracy",
        "summary_accuracy",
        "goal_accomplishment",
    ]

    modal_html = []
    table_rows = []
    modal_counter = [0]

    def make_modal(res, variant, trace_name):
        mid = f"modal-{modal_counter[0]}"
        modal_counter[0] += 1
        sc = res.get("score")
        sl = f"{sc}/10" if isinstance(sc, int) else "N/A"
        det = res.get("traceback", [])
        rw = res.get("raw_response") or res.get("raw_evaluation")
        rw_html = (
            f"<details><summary>Raw evaluator response</summary><pre>{escape(str(rw))}</pre></details>"
            if rw else ""
        )

        modal_html.append(f'''<div class="modal-overlay" id="{mid}">
<div class="modal"><div class="modal-header"><h2>{escape(str(res.get("metric","")).replace("_"," ").title())}</h2>
<button onclick="closeModal('{mid}')">&times;</button></div>
<div class="meta"><b>Variant:</b> {escape(str(variant))} | <b>Trace:</b> {escape(str(trace_name))} | <b>Score:</b> {sl}</div>
<h3>Reasoning</h3><p>{escape(str(res.get("reasoning","—")))}</p>
<h3>Details / Traceback</h3>{render_detail_table(det if isinstance(det, list) else [det])}
{rw_html}</div></div>''')
        return mid, sl

    for (var, tname, tid), mres in traces.items():
        cells = []
        for m in metric_order:
            r = next((x for x in mres if x.get("metric") == m), None)
            if r:
                mid, sl = make_modal(r, var, tname)
                sc = r.get("score")
                cls = "high" if isinstance(sc, int) and sc >= 8 else "medium" if isinstance(sc, int) and sc >= 5 else "low"
                cells.append(f'<td><button class="score {cls}" onclick="openModal(\'{mid}\')">{sl}</button></td>')
            else:
                cells.append('<td class="na">N/A</td>')
        scs = [x.get("score") for x in mres if isinstance(x.get("score"), int)]
        avg = round(sum(scs) / len(scs), 2) if scs else "N/A"
        acls = "high" if isinstance(avg, (int, float)) and avg >= 8 else "medium" if isinstance(avg, (int, float)) and avg >= 5 else "low"
        table_rows.append(
            f"<tr><td>{escape(str(var))}</td><td>{escape(str(tname))}</td>{''.join(cells)}<td><span class=\"avg {acls}\">{avg}</span></td></tr>"
        )

    mheads = "".join(f"<th>{m.replace('_',' ').title()}</th>" for m in metric_order)
    doc = f'''<!doctype html>
<html><head><meta charset="utf-8"><title>LangSmith Evaluation Report</title>
<style>
body{{font-family:Inter,Arial,sans-serif;background:#0f172a;color:#e2e8f0;padding:32px}}
table{{border-collapse:collapse;width:100%;background:#1e2937}}
th,td{{padding:11px 13px;border-bottom:1px solid #334155}}
th{{background:#334155;color:#bae6fd}}
.score{{border:0;border-radius:5px;padding:5px 11px;font-weight:700;color:#0f172a;cursor:pointer}}
.high{{background:#4ade80}}.medium{{background:#fbbf24}}.low{{background:#f87171}}
.na{{color:#64748b;font-style:italic}}
.avg{{font-weight:700;padding:4px 10px;border-radius:4px}}
.modal-overlay{{display:none;position:fixed;inset:0;background:rgba(15,23,42,.85);padding:40px;overflow:auto}}
.modal-overlay.open{{display:block}}
.modal{{max-width:1080px;margin:auto;background:#1e2937;border:1px solid #475569;border-radius:10px;padding:22px}}
.modal-header{{display:flex;justify-content:space-between}}
.modal-header button{{font-size:26px;background:none;border:0;color:#e2e8f0;cursor:pointer}}
.detail-scroll{{overflow:auto;max-height:460px}}
.detail-scroll table{{min-width:760px}}
pre{{white-space:pre-wrap;word-break:break-word;background:#0c1322;padding:12px;border-radius:6px}}
.empty{{color:#9ca3af}}
</style></head><body>
<h1>LangSmith Evaluation Report</h1>
<p style="color:#64748b">One row per trace • Click any score to see detailed traceback</p>
<table>
<thead><tr><th>Variant</th><th>Trace</th>{mheads}<th>Avg</th></tr></thead>
<tbody>{''.join(table_rows)}</tbody>
</table>
{''.join(modal_html)}
<script>
function openModal(id){{document.getElementById(id).classList.add("open")}}
function closeModal(id){{document.getElementById(id).classList.remove("open")}}
document.addEventListener("click",e=>{{if(e.target.classList.contains("modal-overlay"))e.target.classList.remove("open")}});
document.addEventListener("keydown",e=>{{if(e.key==="Escape")document.querySelectorAll(".modal-overlay.open").forEach(el=>el.classList.remove("open"))}});
</script></body></html>'''

    Path(output_file).write_text(doc, encoding="utf-8")
    return output_file


HTML_REPORT_FILE = build_interactive_report(evaluation_results)
print(f"✓ Interactive details report exported to '{HTML_REPORT_FILE}'")

✓ Interactive details report exported to 'evaluation_details_report.html'


## 11. Log Evaluation Results back to LangSmith

Logs the evaluation results as feedback into LangSmith.
- **Trace-level metrics** (`goal_accomplishment`, `trajectory_match`) → attached to root trace
- **Span-level metrics** (`parameter_extraction`, `plan_accuracy`, `tool_call_accuracy`, `summary_accuracy`) → attached to their respective child spans


In [ ]:
# ============================================================
# Log Evaluation Results back to LangSmith
# ============================================================

from langsmith import Client

def get_span_id_by_name(execution_trace, span_name):
    """Recursively find the first span with the given name and return its id."""
    if execution_trace.get("name") == span_name:
        return execution_trace.get("id")
    for child in execution_trace.get("children", []):
        found = get_span_id_by_name(child, span_name)
        if found:
            return found
    return None


def log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces):
    """Log evaluation metrics back to LangSmith as feedback."""
    print("=" * 70)
    print("Logging Evaluation Results to LangSmith")
    print("=" * 70)

    # Group results by trace
    results_by_trace = {}
    for res in evaluation_results:
        tid = res.get("trace_id")
        if tid not in results_by_trace:
            results_by_trace[tid] = []
        results_by_trace[tid].append(res)

    logged_count = 0

    for idx, (root_run, exec_trace) in enumerate(zip(root_runs, execution_traces)):
        trace_id = str(root_run.id)
        trace_results = results_by_trace.get(trace_id, [])

        if not trace_results:
            continue

        print(f"\nLogging feedback for trace {idx+1}: {root_run.name}")

        for result in trace_results:
            metric = result.get("metric")
            score = result.get("score")
            reasoning = result.get("reasoning", "")

            if score is None:
                continue

            # Normalize score to 0-1 range for LangSmith (optional but common)
            normalized_score = round(score / 10.0, 4)

            # ============================================================
            # TRACE-LEVEL METRICS → attach to root
            # ============================================================
            if metric in ["goal_accomplishment", "trajectory_match"]:
                try:
                    langsmith_client.create_feedback(
                        key=metric,
                        score=normalized_score,
                        trace_id=trace_id,
                        comment=reasoning[:500] if reasoning else None,
                        source_info={
                            "evaluator": "llm-judge",
                            "raw_score": score,
                            "max_score": 10
                        }
                    )
                    print(f"  ✓ Trace-level: {metric} = {score}/10 → root trace")
                    logged_count += 1
                except Exception as e:
                    print(f"  ✗ Failed to log {metric}: {e}")

            # ============================================================
            # SPAN-LEVEL METRICS → attach to specific spans
            # ============================================================
            elif metric == "parameter_extraction":
                span_id = get_span_id_by_name(exec_trace, "Parameter Extraction Agent")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="parameter_extraction_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: parameter_extraction → Parameter Extraction Agent")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "plan_accuracy":
                span_id = get_span_id_by_name(exec_trace, "Supervisor: Plan & Route")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="plan_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "tool_call_accuracy":
                # Attach to Research Agent and Analysis Agent as examples
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"tool_call_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: tool_call_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

            elif metric == "summary_accuracy":
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"summary_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: summary_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

    print("=" * 70)
    print(f"Successfully logged {logged_count} feedback entries to LangSmith.")
    print("=" * 70)


# Run the logging
log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces)


Logging Evaluation Results to LangSmith

Logging feedback for trace 1: Supervisor Agent
  ✓ Trace-level: trajectory_match = 8/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis Agent
  ✓ Span-level: tool_call_accuracy → Report Writer Agent
  ✓ Span-level: summary_accuracy → Research Agent
  ✓ Span-level: summary_accuracy → Analysis Agent
  ✓ Span-level: summary_accuracy → Report Writer Agent
  ✓ Trace-level: goal_accomplishment = 4/10 → root trace

Logging feedback for trace 2: Supervisor Agent — variant_A_poor_parameter_and_plan
  ✓ Trace-level: trajectory_match = 8/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis Age